In [1]:
import pyterrier_colbert
import pyterrier as pt

In [2]:
index = pt.Artifact.from_hf("pyterrier/msmarco_psg_v1.colbertv2", 
    plaid_mode=True, ncells=4,
    centroid_score_threshold=0.4, ndocs=4096)

/opt/miniconda3/envs/colbert2/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


[Jun 22, 16:52:29] #> Loading codec...
[Jun 22, 16:52:29] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Jun 22, 16:52:31] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Jun 22, 16:52:31] #> Loading IVF...
[Jun 22, 16:52:32] #> Loading doclens...


100%|██████████| 354/354 [00:00<00:00, 536.73it/s]


[Jun 22, 16:52:34] #> Loading codes and residuals...


100%|██████████| 354/354 [00:15<00:00, 22.93it/s]


In [3]:
index.end_to_end()

pt.apply.by_query()

In [4]:
index.end_to_end().search("what are chemical reactions?").head(2)


#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: what are chemical reactions?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([ 101,    1, 2054, 2024, 5072, 9597, 1029,  102,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



,qid,query,docno,score,rank
0,1,what are chemical reactions?,661396,27.687500,0
1,1,what are chemical reactions?,8572191,27.546875,1


In [5]:
index.plaid_prf_end_to_end(top_psg=3, top_exp=14, beta=0.7)

(pt.apply.by_query() >> pt.apply.by_query())

In [6]:
# TREC-DL 2019 pt.Experiment:
from pyterrier.measures import *
pt.Experiment(
    {
        "PLAID" : pt.rewrite.tokenise() >> index.end_to_end(), 
        "PLAID-PRF": pt.rewrite.tokenise() >> index.plaid_prf_end_to_end(top_psg=3, top_exp=14, beta=0.7)
    },
    pt.get_dataset("msmarco_passage").get_topics("test-2019"),
    pt.get_dataset("msmarco_passage").get_qrels("test-2019"),
    eval_metrics=[nDCG@10], batch_size=1, verbose=True
)

Java started (triggered by tokenise) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


pt.Experiment:   0%|          | 0/400 [00:00<?, ?batches/s]

,name,nDCG@10
0,PLAID,0.738293
1,PLAID-PRF,0.769527
